# 01 — Esplorazione del dataset Kaggle Fraud Detection

## Obiettivi didattici

1. Comprendere la struttura del dataset (~1.5M transazioni, frodi ~0.5%).
2. Visualizzare lo **sbilanciamento delle classi** e i suoi effetti sulle metriche standard.
3. Analizzare la **distribuzione dell'importo** e il legame con l'etichetta di frode.
4. Identificare **pattern temporali** (orario, giorno, mese) associati alle frodi.
5. Esplorare la **distanza geografica** cliente-merchant.

!!! note "Dataset richiesto"
    Il dataset Kaggle (~470MB) NON e' in repo per limiti di GitHub.
    Scaricalo da <https://www.kaggle.com/datasets/kartik2112/fraud-detection>
    e copia `fraudTrain.csv` e `fraudTest.csv` in `data/raw/`.


In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

from fraud_pipeline.data import load_train_test, class_distribution
from fraud_pipeline.features import haversine_km


## Caricamento

Carichiamo i due CSV Kaggle. Il modulo `data.load_train_test` esegue automaticamente: validazione schema, parse del datetime, ordinamento cronologico.

In [ ]:
df_train, df_test = load_train_test()
print(f'Train: {df_train.shape}, frodi={df_train.is_fraud.sum()}')
print(f'Test : {df_test.shape}, frodi={df_test.is_fraud.sum()}')
df_train.head(3)


## Distribuzione classi: sbilanciamento estremo

La classe positiva (`is_fraud=1`) rappresenta una frazione molto piccola dei dati. Conseguenze pratiche:

- **Accuracy non utile**: predire sempre 0 da' >99% accuracy ma 0% recall sulle frodi.
- **Metriche da preferire**: AUC-PR (Average Precision), Recall, F1, F2 sulla classe positiva.
- **Strategie di gestione**: `class_weight='balanced'` o resampling (SMOTE) — vedi notebook 03.


In [ ]:
dist = class_distribution(df_train.is_fraud)
print('Train class distribution:')
for k, v in dist.items():
    print(f'  {k}: {v}')

fig, ax = plt.subplots(figsize=(6, 4))
counts = df_train.is_fraud.value_counts()
ax.bar(['Legit (0)', 'Fraud (1)'], counts.values, color=['#4C72B0', '#C44E52'])
for i, c in enumerate(counts.values):
    ax.text(i, c, f'{c:,}', ha='center', va='bottom')
ax.set_yscale('log')
ax.set_title(f'Distribuzione classi (Train) — frodi: {100*dist["positive_rate"]:.3f}%')
plt.show()


## Distribuzione dell'importo

Le frodi tipicamente si concentrano in due cluster: **importi piccoli** (< $1, frodi di test della carta) e **importi medi-grandi** (massimizzazione del danno prima del blocco).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label, ax in zip([0, 1], axes):
    sub = df_train.loc[df_train.is_fraud == label, 'amt']
    ax.hist(np.log1p(sub), bins=60, edgecolor='black',
            color='#4C72B0' if label == 0 else '#C44E52', alpha=0.85)
    ax.set_title(f'log1p(amt) — {"Legit" if label == 0 else "Fraud"} '
                 f'(median ${sub.median():.2f})')
    ax.set_xlabel('log1p(amt)')
fig.tight_layout(); plt.show()


## Pattern temporali

Decomponiamo il timestamp in ora, giorno della settimana, mese e guardiamo il fraud rate condizionato. Frodi tipicamente piu' frequenti di notte (carte rubate usate quando il proprietario dorme).

In [ ]:
tmp = df_train.assign(
    hour=df_train.trans_date_trans_time.dt.hour,
    dayofweek=df_train.trans_date_trans_time.dt.dayofweek,
    month=df_train.trans_date_trans_time.dt.month,
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['hour', 'dayofweek', 'month']):
    rate = tmp.groupby(col)['is_fraud'].mean() * 100
    ax.bar(rate.index, rate.values, color='#C44E52')
    ax.set_title(f'Fraud rate per {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Fraud rate (%)')
fig.tight_layout(); plt.show()


## Analisi per categoria

Le 14 categorie di merchant non hanno tutte lo stesso profilo di rischio.

In [ ]:
cat_stats = (df_train.groupby('category')['is_fraud']
             .agg(['mean', 'count']).rename(columns={'mean': 'fraud_rate'}))
cat_stats['fraud_rate'] *= 100
cat_stats = cat_stats.sort_values('fraud_rate', ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(cat_stats.index, cat_stats.fraud_rate, color='#C44E52')
ax.set_xlabel('Fraud rate (%)')
ax.set_title('Fraud rate per categoria merchant')
for i, v in enumerate(cat_stats.fraud_rate):
    ax.text(v, i, f' {v:.2f}%', va='center')
plt.tight_layout(); plt.show()
cat_stats


## Distanza geografica cliente-merchant

Calcoliamo la distanza Haversine fra coordinate cliente e merchant. Frodi spesso avvengono a distanze maggiori (carte rubate / card-not-present con localizzazioni anomale).

In [ ]:
df_train['distance_km'] = haversine_km(
    df_train.lat, df_train.long, df_train.merch_lat, df_train.merch_long,
)
fig, ax = plt.subplots(figsize=(9, 4))
for label, color in [(0, '#4C72B0'), (1, '#C44E52')]:
    sns.kdeplot(
        df_train.loc[df_train.is_fraud == label, 'distance_km'].clip(upper=200),
        ax=ax, label=f'is_fraud={label}', color=color, lw=2,
    )
ax.set_xlabel('distance_km (clipped a 200)')
ax.set_title('Distanza geografica per classe')
ax.legend(); plt.show()


## Conclusioni dell'EDA e implicazioni per il modeling

| Osservazione | Implicazione |
|---|---|
| Frodi ~0.5% | Metrica primaria: **AUC-PR**, non accuracy ne' AUC-ROC. |
| Importo bimodale | Costruire feature `is_small_amt`, `log_amt`. |
| Pattern orario | Costruire `hour`, `is_night`, `is_weekend`. |
| Categoria predittiva | Tenere `category` come feature OneHot. |
| Distanza geografica | Calcolare `distance_km` con Haversine. |
| Sequenzialita' temporale | Split train/test cronologico, walk-forward CV. |

-> Procedi al notebook **02_feature_engineering**.
